# 03 — Справочник и проверка соответствия

Как код проверяет операции по профилю компании до и после ответа модели. GPU не нужен.

In [ ]:
import os, sys
if not os.path.exists("src/acctltm"):
    !git clone https://github.com/serge3emskov/accountant-llm-assistant.git
    os.chdir("accountant-llm-assistant")
!pip install -q pydantic rapidfuzz
sys.path.insert(0, "src")

In [ ]:
from acctltm import (AccountingCatalog, Normalizer, MemoryStore, MemoryWriter,
                     ComplianceChecker)
from acctltm.compliance import RuleSet
from acctltm.devcheck import load_jsonl

cat = AccountingCatalog()
rules = RuleSet.load()
print("правил:", len(rules.rules), "классов:", len(rules.classes))

## Профиль компании из памяти

In [ ]:
norm = Normalizer(cat)
store = MemoryStore()
writer = MemoryWriter(store, norm)
writer.write("u", {"facts": [
    {"section": "company_profile", "attribute": "regime", "value": "УСН", "evidence": "Мы на УСН"},
    {"section": "company_profile", "attribute": "usn_object", "value": "доходы", "evidence": "УСН доходы"},
    {"section": "practices", "attribute": "cash_settlements", "value": "да", "evidence": "Расчёты наличными"},
    {"section": "contracts", "name": "договор поставки с Ромашкой", "status": "active",
     "amount": 150000, "evidence": "договор поставки с Ромашкой на 150 тысяч"},
    {"section": "counterparties", "name": "Ромашка", "status": "active", "evidence": "Работаем с Ромашкой"},
], "section_statuses": []}, "d", dialog_text="Мы на УСН доходы. Расчёты наличными. договор поставки с Ромашкой на 150 тысяч. Работаем с Ромашкой.")

chk = ComplianceChecker(cat)
prof = chk.profile(store, "u")
print(prof.as_dict())

## Проверка до ответа (ограничения по профилю)

In [ ]:
rep = chk.report(prof, "Можно оплатить Ромашке наличными 150 тысяч и принять расходы?")
for a in rep.alerts:
    print("ALERT:", a.render())
for r in rep.restrictions:
    print("RESTR:", r.render())

## Проверка после ответа

Если модель не назвала причину предупреждения, код дописывает его.

In [ ]:
naive = "Можно оплатить наличными и принять расходы на аренду."
post = chk.post_check(naive, prof, rep.question_entities)
print("дописано кодом:", [a.rule_id for a in post.appended])
print(post.text)

good = "При объекте УСН доходы расходы не уменьшают налог; наличные свыше 100000 по договору запрещены."
post2 = chk.post_check(good, prof, rep.question_entities)
print("\nучтено моделью:", [a.rule_id for a in post2.covered])

## Кейсы комплаенса

In [ ]:
cases = load_jsonl("data/compliance/cases.jsonl")
for c in cases:
    st = MemoryStore(); w = MemoryWriter(st, norm)
    setup = c["setup"]
    dt = " ".join(f["evidence"] for f in setup["facts"]) + " " + " ".join(s["evidence"] for s in setup["section_statuses"])
    w.write("u", setup, c["id"], dialog_text=dt)
    p = chk.profile(st, "u", c["subject"])
    r = chk.report(p, c["question"])
    actual = {a.rule_id for a in r.alerts}
    ok = set(c["expect_rules"]) <= actual
    print(c["id"], "OK" if ok else "FAIL", c["title"])

## Вывод

Всё, что можно проверить детерминированно (лимиты, режимы, риск-флаги контрагентов, первичка), проверяет код — и до, и после ответа модели.